# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 25  
**Kaggle challenge:**  `Deep learning` \
**Kaggle team name (exact):** "Image-inativi"  

**Author 1 (SCIPER):** Chiara Evangelisti (368672)   
**Author 2 (SCIPER):** Elisa Ferrara (371064)  
**Author 3 (SCIPER):** Francesco Maglie (378056)  

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

1. [Introduction](#introduction)
2. [Data Augmentation](#data-augmentation)
    - 2.1 [Roboflow](#roboflow)  
    - 2.2 [Masked](#masked)  
3. [Pre-processing](#pre-processing)
    - 3.1 [Filters](#filters)  
    - 3.2 [Color modifications](#colors)
4. [Model](#model)
    - 4.1 [Initial model](#initial)  
    - 4.2 [Architecture modification](#architecture)
5. [Training](#training)
    - 5.1 [Training loop and optimizer](#loop)  
    - 5.2 [Losses](#losses)
    - 5.3 [Validation](#validation)
7. [Post-processing](#post-processing)
    - 6.1 [Connected components identification](#connected)  
    - 6.2 [Area- based separations](#area)
8. [Limitation and final considerations](#testing--inference)
9. [References](#references)


## 1) Introduction
<a id="introduction"></a>

The goal of this project is to develop a deep learning papeline capable of accurately **recognizing and counting different types of chocolates**, known in advance, in images.

The dataset consists of multiple chocolates distributed on background of increasing complexity and aggregated in clusters and positions of different type.  Each image is accompanied by a weak annotation: indicating the total count of chocolates of each type present in the image. Additionally, a small subset of reference images is provided, showing the appearance of each chocolate class.

This project explores a custom semantic segmentation pipeline using a modified U-Net architecture trained with a combination of loss functions and some post -processing to perfect the segmentation mask and count the number of instances for each class found by the network.

Here's a **schematic representation of the training pipeline** implemented, which will be detailed in the following sections:
![alt text](Images/scheme.png)

## 2) Data Augmentation
<a id="data-augmentation"></a>

Techniques used to increase dataset diversity (e.g., flipping, rotation, brightness, color jitter, Gaussian blur, etc.).

### 2.1 Roboflow
<a id="roboflow"></a>

### 2.2 Masked Augmentation
<a id="masked"></a>

To improve the robustness of our segmentation pipeline, we introduced an additional fine-tuning step that targets failure cases, particularly those involving **occlusions**. We observed that occluded chocolates—partially hidden by other elements—were not being segmented accurately. To address this, we fine-tuned our U-Net model for 20 more epochs on a specialized, **augmented dataset** where parts of the chocolates were artificially occluded using a custom transform.

The core of this augmentation is a handcrafted **CutoutTransform**, implemented using PyTorch and NumPy functions. The transform randomly removes square patches from chocolate regions based on the segmentation mask, simulating occlusions in a realistic way. Specifically, for each training image and its corresponding mask:
- Non-zero (i.e., foreground) pixels are identified.
- A predefined number of random regions (holes) are selected, centered on foreground pixels.
- These regions are zeroed out in both the image and the mask, effectively "cutting off" parts of the object.

This process is applied dynamically to generate new training examples, which are saved into a dedicated folder structure (`CutoutImages/` and `CutoutMasks/`). A list of the processed filenames is also recorded for reproducibility (`train_cutoff.txt`). This targeted data augmentation aims to help the model generalize better to real-world scenarios where chocolates may be partially obstructed. In the image below you can see an example of cutout image.

![cutout](Images/cutout.png)




## 3) Pre-processing
At the early stages of our project, we tested out the Unet architecture (see Section 4) on our dataset, and we observed that it would assign multiple classes to a single chocolate or it would confuse similar chocolates (for instance Comptesse and Jelly White), as you can see in the following pictures.
<a id="pre-processing"></a>
![alt text](Images/before_filter1.jpg)

![alt text](Images/before_filter2.jpg)

This is likely due on one side to the model behavior which relies heavily on color and texture details to distinguish different chocolates instead of focusing on a broader scale. So this behavior would fail in cases where for instance shadows make two different chocolates looking similar. Therefore to ease the task and force the model to focus on less detaield information and more on the overall object color/pattern we decided to add some pre-processing to the training images coming from augmentation. Before undergoing the transformations mentioned in the next sub-sections, all images are resized to have the longest side 512 while maintaining the aspect ratio, to avoid distortions that could hurt the segmentation performance, while having computationally cheaper images.

### 3.1 Filters
<a id="filters"></a>
To remove potential noise and remove very fine-grained details, our initial tentative was to apply a **Gaussian filter** to the input images to smooth them out a bit. While it did improve our performance and worked well when the background was much different from the chocolates to be recognized, its performance was poorer when the background had similar colors to the chocolates (for instance the brown/beige t-shirt). Therefore we decided to use instead the **Bilateral filter**.\
As explained in [1] the Gaussian blurring has the folling formulation:

$$GB[I]_p = \sum_{q \in S} G_\sigma(\|p - q\|) I_q$$

Here, $ GB[I]_p $ is the result at pixel $ p $, and the RHS is the result of the convolution with the Gaussian kernel, $ I_q $ is the intensity at pixel $ q $.\
The bilateral filter on the other end can be expressed as:

$$
BF[I]_p = \frac{1}{W_p} \sum_{q \in S} G_{\sigma_s}(\|p - q\|) \cdot G_{\sigma_r}(|I_p - I_q|) \cdot I_q
$$

Where:

- $ \frac{1}{W_p} $ — **Normalization factor** , sum of weigths
- $ G_{\sigma_s}(\|p - q\|) $ — **Spatial weight** (depends on the pixel distance) , w/ $\sigma_s$ denoting neighborhood size
- $ G_{\sigma_r}(|I_p - I_q|) $ — **Range weight** (depends on intensity difference) , w/ $\sigma_r$ denoting minimum edge amplitude, Gaussian blur at $\sigma_r = \infty$
- $ I_q $ — Intensity at pixel $q$

The bilateral filter smooths images **while preserving edges** by combining ensuring that only those pixels with intensity values similar to that of the central pixel undergoing convolution are considered for blurring, while sharp intensity changes are maintained.\
 Let's consider a pixel $p$ that lies near a sharp edge, where its intensity $I_p$ differs significantly from some of its neighbors $q$. For neighboring pixels $q$ that lie on the same side of the edge (i.e., they have a similar intensity), the intensity difference $|I_p - I_q|$ is close to zero. As a result, the range weight $G_{\sigma_r}(|I_p - I_q|) \approx 1$, allowing those pixels to contribute significantly to the final average. In contrast, for neighbors $q$ that lie across the edge (with a very different intensity), the difference $|I_p - I_q|$ becomes large. Consequently, $G_{\sigma_r}(|I_p - I_q|) \approx 0$, which  downweights or  excludes these pixels from the smoothing. \
The set of parameters which we have found to work best are:
- $d = 9$ kernel size
- $\sigma_{\text{color}} = 45$ Range filter
- $\sigma_{\text{space}} = 45$ Spatial filter
Indeed we tested with initially higher parameters, but that lead to a signficant loss of information so that the model ended up segmenting external objects (like the wallet) as chocolates. By reducing then in different training, this set was empirically found.


### 3.2 Color modifications
<a id="colors"></a>

Another type of preprocessing we decided to apply focuses on the **color properties** of the input images. Specifically, we used the `ColorJitter` transformation with controlled variations in **brightness**, **contrast**, and **saturation**:

```python
transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2)
```
This technique introduces random but realistic changes to the image appearance, helping the model become more robust to lighting conditions and visual variability that may occur in real-world scenarios. The decision to include color jittering was inspired by its common use in data augmentation for image classification and segmentation tasks, where improving generalization to unseen lighting and camera settings is critical. In our case, this is particularly useful for chocolate detection, where reflections and lighting inconsistencies can vary significantly across samples. In particular, an image which was causing some trouble to the segmentation network is the on eshowed below. You can visualize in the image 3 possible color mdifications. Jitter is applied randomly and in the image we show 3 different possibilities of the output of the modification.

![Jitter](Images/jitter.png)



## 4) Model
<a id="model"></a>

Model architecture definition (e.g., U-Net, ResNet-UNet, etc.), number of parameters, and discussion of key layers.

In this section we will describe the choice of the segmentation network we designed, starting from the goal we will delve deeper into the state-pf.the-art models that we took inspiration from and we will conclude dissectionating the various layers of the model.

**GOAL**: Image segmentation.

Image Segmentation is a computer vision task that involves dividing an image into meaningful regions or segments. The goal is to assign a label to every pixel in the image, making it easier to analyze its content.
There are two main types of segmentation:
- Semantic segmentation: Assigns each pixel to a class label (e.g., car, road, tree), but does not distinguish between different instances of the same class.
- Instance segmentation: Goes a step further by distinguishing between individual objects of the same class (e.g., two different cars in the same image).

In our project, we focus on semantic segmentation, as the objective of this step is to assign each pixel in the image to one of 13 predefined classes, without needing to differentiate between multiple objects of the same category. The counting step will be done in the post-processing.

As a first step in designing our custom segmentation network, we conducted a study of the most widely used segmentation architectures. The aim was to identify well-established models that could serve as a foundation or source of inspiration for our own design.

In our pipeline, the segmentation module at test time takes as input a pre-processed RGB image (as previously described) and outputs a mask where each pixel is assigned a value from 0 to 13. Here, 0 corresponds to the background class, while values from 1 to 13 represent different target classes. That said, at training time the ground-truth is represented by the segmentation masks generated by hand with Roboflow (as explained before).

**SOTA SEGMENTATION NETWORKS**

The following table summarizes our research on potential segmentation networks to use as inspiration. In selecting suitable architectures, we considered not only the original purpose and strengths of each network, but also their number of parameters. Since our project has a strict constraint of using no more than 12 million parameters, starting from a large network could make it difficult to reproduce strong performance within such a limited capacity.

| **Network**        | **Advantages**                                                        | **Limitations**                                                  | **Parameters (approx.)**           | **Typical Applications**               | **Output Type**                         |
|--------------------|----------------------------------------------------------------------|------------------------------------------------------------------|------------------------------------|----------------------------------------|------------------------------------------|
| **FCN (VGG/ResNet)** | Simple architecture; end-to-end trainable; flexible backbone         | Coarse segmentation outputs without skip connections             | VGG: ~134M / ResNet50: ~23M        | General semantic segmentation          | Semantic segmentation map                |
| **U-Net**          | High accuracy with few training samples; skip connections preserve details | Can be less scalable for very large datasets                     | ~31M                               | Biomedical image segmentation          | Semantic segmentation map                |               |
| **Mask R-CNN**     | Provides instance segmentation; accurate bounding boxes and masks     | Slower; more complex architecture                                | ~44M (ResNet50 backbone)           | Instance segmentation (e.g., COCO)     | Instance masks + bounding boxes         |
| **YOLO (v3–v8)**   | Very fast; real-time object detection; extensible to segmentation     | Not originally designed for segmentation; needs adaptation       | Varies (YOLOv5s: ~7M, YOLOv5m: ~21M) | Real-time detection, lightweight tasks | Bounding boxes (standard); masks (if extended with segmentation head) |

We chose U-Net for the segmentation step of our chocolate recognition project due to its ability to produce accurate, high-resolution pixel-wise predictions even with limited training data. Its encoder-decoder architecture with skip connections makes it particularly effective for capturing fine details, which is essential for identifying distinct chocolate regions.



### 4.1 Initial model: U-Net
<a id="initial"></a>

**U-Net** is a convolutional neural network architecture specifically designed for **semantic segmentation**. It was originally developed for biomedical image segmentation but has since become popular in many other domains due to its effectiveness, especially when training data is limited.

#### 🔹 General Idea

U-Net follows an **encoder-decoder** structure. The main idea is to extract context (what) from the image using the **encoder**, and then precisely localize (where) objects using the **decoder**. This is achieved by combining low-level spatial information from early layers with high-level semantic information from deeper layers using **skip connections**.


#### 🔹 Architecture Overview

As introduced in [Ronneberger et al., 2015], U-Net has a symmetric "U"-shaped structure, composed of two main parts:
![U-Net Diagram](Images/UNET.jpg)

1. **Contracting Path (Encoder)**  
   - Purpose: Captures **contextual features** and reduces spatial resolution.
   - Composition:
     - Repeated blocks of:
       - Two **3×3 convolution** layers (with ReLU)
       - One **2×2 max pooling** (stride 2) to downsample
     - The number of feature channels doubles at each downsampling step.

2. **Expanding Path (Decoder)**  
   - Purpose: Restores **spatial resolution** and enables precise localization.
   - Composition:
     - Repeated blocks of:
       - **Up-convolution** (transposed convolution) to upsample
       - **Concatenation** with the corresponding encoder feature map (skip connection)
       - Two **3×3 convolution** layers (with ReLU)
     - The number of feature channels halves at each upsampling step.

3. **Final Layer**  
   - A **1×1 convolution** maps the output to the desired number of segmentation classes, producing a **per-pixel class prediction**.


#### 🔹 Why Skip Connections?

Skip connections allow the decoder to reuse **high-resolution features** from the encoder, improving the network’s ability to **preserve spatial details** and handle **fine structures**, which is crucial in pixel-level prediction tasks.


#### 🔹 Summary

U-Net’s structure allows it to perform well even with small datasets and limited training data. It is efficient, easy to train end-to-end, and produces high-resolution segmentation maps — making it ideal for tasks that require **detailed pixel-level accuracy**.

We used as starting point this implementation of u-net: https://github.com/clemkoa/u-net/tree/master .


### 4.2 Architecture modifications
<a id="initialmodel"></a>
The basic structure for our model was taken from [3] which is an implementation based on the original paper [2], with only two minors differences: no padding in the the pooling layers to make handling dimensions easier and no class weights in softmax to handle class imbalance. 

We performed some modificationson the architecture of the original Unet for two main reasons: first the original architecture had around 31 millions of parameters, which does not respect the limit of 12 millions imposed by the challenge, then as mentioned in the previous section, the model often identified different chocolates inside a single of them.

The main differences with the reference implementation are:

- **reduced depth**: 3 downsampling and upsampling layers instead of 4 to respect the parameters limit. We chose to have a shallower network instead of maintaining the depth and reducing the number of hidden units per layer because the former allows to preserve more the global structure and information , which for our application is more important than fine, texture-level details which are instead the focus of the latter. This of course results also in a reduced number of channels at the bottleneck.
- **units at output layer**: the model was implemented for a binary segmentation task so the output dimension was just 2, in our case we have 14 output units for the 13 chocolate classe and the background.
- **dilated convolution**: it a technique that allows to expand the receptive field of the filter without increasing the number of parameters or the amount of computation. This is achieved by inserting gaps (zeros) between the elements of the convolutional kernel, so that some pixels are skipped, as it can be seen in the following figure.
![alt text](Images/dilated_conv.png)
This allows to capture a broader context without additional resources needed and to maintain at the same time the spatial dimension of the features (as long as padding compensates appropriately). A dilation of size 2 has been used in the convolutional layers of the network.

The resulting model has **7703822 parameters**. 


## 5) Training
<a id="training"></a>

Training loop, optimizer, loss functions (e.g., CrossEntropy, Dice), and learning rate strategy.


### 5.1 Training loop and optimizer
<a id="loop"></a>

#### Training
The training pipeline is designed to optimize a U-Net model for semantic segmentation using a combination of data preprocessing, customized loss functions, and robust training practices.

#### Optimizer
We use the **RMSprop** optimizer with a learning rate of `1e-4`, a weight decay of `1e-8`, and a momentum of `0.9`. RMSprop is well-suited for image tasks with noisy gradients and offers a good trade-off between speed and stability. This optimizer and optimization parameters are the ones used in the u-net implementation we took inspiration from (https://github.com/clemkoa/u-net/tree/master).


### 5.2 Losses
<a id="losses"></a>

In the reference implementation in [3] the standard cross entropy loss is used, but the issues experimented in the initial tests (see Section 2), other losses were explored, here [5] an overwiew on different losses often used in (medical) imaging.\
In particular we used a combination of these losses:
- **(Weigthed) cross entropy loss:** based on KL divergence, which measures dissimilarity between two distributions. Tries to match the predicted distribution to the ground truth.  
$$\mathcal{L}_{CE} = - \sum_{c=1}^{C} y_c \log(\hat{y}_c)$$
Its weighted modification assigns different weights to the different classes, usually the least represented class are given higher weights to compensate for class imbalance.
$$\mathcal{L}_{WCE} = - \sum_{c=1}^{C} w_c \, y_c \log(\hat{y}_c)$$
Where $w_c$ is the weight for class $c$, usually set as the **inverse of class frequency**.

- **Dice  loss:** prioritizes overlap between predictions and targets , by rewarding the model for correcty predicting true positives and penalizing false positive and false positives wehn comparing two binary classifications of an image. This property makes it suitable for handling datasets where certains regions dominate.
![DSC](Images/diceloss.jpeg)
$$\text{Dice similarity coefficient} = \frac{2 \sum_i p_i g_i}{\sum_i p_i + \sum_i g_i + \epsilon}$$
 where $p_i$: predicted probability at pixel $i$  and $g_i$: ground truth label at pixel $i$ (usually 0 or 1)

$$\mathcal{L}_{Dice} = 1 - \text{Dice similarity coefficient}$$
- **Tversky loss:** designed to measure the dissimilarity between two sets, controlling the tradeoff between false positives and false negatives, to customize the sensitivity of the loss to different types of errors.
![TI](Images/twerky.jpeg)
$$\text{Tversky index} = \frac{\sum_i p_i g_i}{\sum_i p_i g_i + \alpha \sum_i p_i (1 - g_i) + \beta \sum_i (1 - p_i) g_i + \epsilon}$$ 
$$\mathcal{L}_{Tversky} = 1 - \text{Tversky index}$$
Adjust $\alpha$ and $\beta$ to penalize false positives vs. false negatives differently:
- $\alpha > \beta$ → penalize false positives more
-  $\beta > \alpha$ → penalize false negatives more


We tested different uses and combinations of this loss:
1. **Just CE loss**: lead to problems discussed in Section 2
2. **Combination of CE and DICE**: each loss is computed and then a linear combination is taken. First $L = 0.7*CE +0.3*DICE$ was tested, but actually best performance was found with $L =0.5*CE +0.5*DICE$\
However, as it can be seen in the following picture (which was obtained in combination of Gaussian blur and random erasing as preprocessing), least represented class in terms of pixel frequency were hard to identify. 
![alt text](Images/ce_unbalanced.jpg)
3. **Combination of WCE and DICE**: to compensate for class imbalance, weights were computed as the inverse of the class frequencies. The class frequencies were defined in terms of pixel frequency (and not instance frequency) across all the augmented images in the training set, then inversed and normalized to sum up to 1. This allowed to solve the mentioned problem
4. **Combination of WCE and TVERSKY**: despite having a model with good performance, the areas of the mask detected is a bit smaller compared to the ground truth masks, and this can be an issue when counting (see Section 6). To compensate for that and for cases when in much clustered groups a chocolate was mixed, we decided to substitute the DICE LOSS with the Twerky one, with $\alpha = 0.35$ and $\beta = 0.65$, which is suitable for our use case, where missing objects is more critical than detecting slight excess. 

### 5.3) Validation
<a id="validation"></a>

During training, model performance is monitored using a **validation loop** that runs at the end of each epoch. In this phase, the model is switched to evaluation mode using `model.eval()`, which disables dropout and batch normalization updates for consistent results. Additionally, gradient computation is turned off using `torch.no_grad()` to reduce memory usage and speed up inference.

The validation loop iterates over the entire validation dataset (`val_loader`). For each batch, the model predictions are compared to the ground truth using two loss components: **Cross Entropy Loss** and **Dice Loss**. The final validation loss is computed as the average of these two components, weighted equally:
 $$
 Validation Loss = 0.5 * CrossEntropyLoss + 0.5 * DiceLoss
 $$

This combined loss provides a balanced measure: Cross Entropy captures pixel-level classification accuracy, while Dice Loss evaluates the spatial overlap between predicted and true segmentations. The average validation loss across all batches is returned and used to track model performance and select the best-performing model during training.

The image below shows the training and validation curve for the best model we trained.

![loss_curve](Images/loss_curve.png)




## 6) Post-processing
<a id="post-processing"></a>

### 6.1 Connected components identification
<a id="connected"></a>

### 6.2 Area- based separation
<a id="connected"></a>

## 7) Limitation and final considerations
<a id="testing--inference"></a>

Running the trained model on test images, visualization of predictions, and export of results (e.g., CSV, overlay masks).


## 8) Reference
<a id="references"></a>

[1] GeeksforGeeks, *Python | Bilateral Filtering*. [https://www.geeksforgeeks.org/python-bilateral-filtering/](https://www.geeksforgeeks.org/python-bilateral-filtering/)


[2] Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. In International Conference on Medical Image Computing and Computer-Assisted Intervention (MICCAI), pp. 234–241. [https://arxiv.org/abs/1505.04597](https://arxiv.org/abs/1505.04597)

[3] clemkoa. *U-Net: PyTorch Implementation of U-Net Architecture*. GitHub repository. Available at: [https://github.com/clemkoa/u-net](https://github.com/clemkoa/u-net)

[4]GeeksforGeeks, *Python | Dilated convolution*. [https://www.geeksforgeeks.org/dilated-convolution/](https://www.geeksforgeeks.org/dilated-convolution/
)

[5] Jun Ma. *Loss Functions for Medical Image Segmentation: A Taxonomy*. Medium, 2021. Available at: [https://medium.com/@junma11/loss-functions-for-medical-image-segmentation-a-taxonomy-cefa5292eec0](https://medium.com/@junma11/loss-functions-for-medical-image-segmentation-a-taxonomy-cefa5292eec0)


In [2]:
## YOUR CODE
...